# Huren vs. Kopen — Cumulatieve kosten over de tijd

Vergelijking van de cumulatieve woonlasten bij **huren** versus **kopen** van de woning aan de Willebrordusstraat 135B-04, Rotterdam.

De aannames staan als parameters bovenaan, zodat je makkelijk scenario's kunt doorrekenen.

**Belangrijke conceptuele keuze:** bij kopen tellen we alleen de *werkelijke kosten* mee (rente + heffingen + VvE), **niet** de aflossing — want aflossing is vermogensopbouw, geen verloren geld. Aan het eind verrekenen we de opbouw (aflossing + waardestijging − verkoopkosten − kosten tweede huis) als je zou verkopen en doorverhuizen.


In [1]:
import numpy as np
import matplotlib.pyplot as plt


## 1. Aannames (parameters)

Pas deze waarden aan om scenario's door te rekenen.

In [ ]:
# ---- HUUR ----
HUUR_NU            = 1857.0     # huidige kale huur per maand (euro)
HUUR_STIJGING      = 0.04       # jaarlijkse huurverhoging (4%)

# ---- KOOP: maandelijkse kosten (geen aflossing) ----
# De bruto maandlast is 2353; daarvan is een deel aflossing (vermogensopbouw).
# Voor "kosten" tellen we rente + heffingen + VvE, NIET de aflossing.
KOOPSOM            = 480000.0
HYPOTHEEK          = 480000.0
OVERBIEDEN         = 0.0        # verschil tussen marktwaarde en aankoopprijs (euro);
                                 # als de woning onder marktwaarde gekocht wordt, levert
                                 # dit extra vermogen op bij verkoop. Default 0.

# Jaarlijkse rente uit de doorrekening, volledige looptijd (jaar 1..31).
RENTE_PER_JAAR = {
    1: 10093,  2: 19927,  3: 19569,  4: 19196,  5: 18808,
    6: 18402,  7: 17979,  8: 17538,  9: 17077, 10: 16597,
    11: 16097, 12: 15574, 13: 15030, 14: 14461, 15: 13869,
    16: 13251, 17: 12606, 18: 11934, 19: 11232, 20: 10501,
    21: 9738,  22: 8942,  23: 8112,  24: 7246,  25: 6343,
    26: 5401,  27: 4418,  28: 3394,  29: 2325,  30: 1210,
    31: 172,
}  # euro per jaar (volledige looptijd)

AFLOSSING_PER_JAAR = {
    1: 4025,   2: 8308,   3: 8665,   4: 9038,   5: 9427,
    6: 9833,   7: 10256,  8: 10697,  9: 11157, 10: 11637,
    11: 12138, 12: 12660, 13: 13205, 14: 13773, 15: 14366,
    16: 14984, 17: 15629, 18: 16301, 19: 17002, 20: 17734,
    21: 18497, 22: 19293, 23: 20123, 24: 20989, 25: 21892,
    26: 22834, 27: 23816, 28: 24841, 29: 25910, 30: 27024,
    31: 13945,
}  # euro per jaar

FISCAAL_VOORDEEL_PER_JAAR = {
    1: 4170,   2: 8224,   3: 8067,   4: 7902,   5: 7731,
    6: 7553,   7: 7367,   8: 7173,   9: 6972,  10: 6760,
    11: 6540,  12: 6311,  13: 6071,  14: 5821,  15: 5560,
    16: 5289,  17: 5006,  18: 4711,  19: 4403,  20: 4081,
    21: 3745,  22: 3396,  23: 3031,  24: 2651,  25: 2253,
    26: 1840,  27: 1408,  28: 957,   29: 487,   30: 0,
    31: 0,
}  # hypotheekrenteaftrek per jaar

# ---- KOOP: lokale heffingen + VvE (eerste jaar, per maand) ----
OZB_TARIEF         = 0.000643   # OZB-tarief 2026 Rotterdam (0,0643% van WOZ)
WOZ_START          = 480000.0   # WOZ-waarde bij aankoop (= taxatie)
RIOOL_PER_MND      = 27.0       # vast Rotterdams tarief 2026
VVE_PER_MND        = 200.0      # SCHATTING - opvragen bij VvE
ORV_PER_MND        = 15.0       # overlijdensrisicoverzekering (schatting)

# ---- JAARLIJKSE STIJGING van belastingen/heffingen ----
WOZ_STIJGING       = 0.04       # jaarlijkse stijging WOZ-waarde -> drijft OZB op
HEFFING_STIJGING   = 0.03       # jaarlijkse stijging riool/VvE/ORV (inflatie-achtig)

# ---- EENMALIGE KOSTEN ----
AANKOOPKOSTEN_NU   = 6552.0     # eigen middelen bij aankoop (akte, taxatie, advies, etc.)
OVERDRACHTSBEL_NU  = 0.0        # startersvrijstelling

# ---- WAARDEONTWIKKELING & VERKOOP ----
WAARDESTIJGING     = 0.00       # jaarlijkse waardestijging woning (conservatief)
MAKELAAR_PCT       = 0.0125     # makelaarscourtage bij verkoop (1,25%)

# ---- OVERSTAP NAAR TWEEDE HUIS ----
OVERSTAP_MEEREKENEN = True
TWEEDE_HUIS_PRIJS   = 600000
OVERDRACHTSBEL_PCT  = 0.02
NIEUWE_FIN_KOSTEN   = AANKOOPKOSTEN_NU

HORIZON_JAREN      = 5


## 2. Hulpfuncties

We berekenen voor elk jaar de cumulatieve *kosten* (verloren geld) van huren en kopen.

In [ ]:
def huur_kosten_cumulatief(jaren):
    """Cumulatieve huurkosten; huur stijgt elk jaar met HUUR_STIJGING."""
    cum = []
    totaal = 0.0
    for j in range(1, jaren + 1):
        jaarhuur = HUUR_NU * 12 * (1 + HUUR_STIJGING) ** (j - 1)
        totaal += jaarhuur
        cum.append(totaal)
    return np.array(cum)


def koop_jaarrente(j):
    return RENTE_PER_JAAR.get(j, 0)


def koop_jaarfiscaal(j):
    return FISCAAL_VOORDEEL_PER_JAAR.get(j, 0)


def heffingen_jaar(j):
    """Jaarlijkse heffingen in jaar j."""
    woz = WOZ_START * (1 + WOZ_STIJGING) ** (j - 1)
    ozb = woz * OZB_TARIEF
    overige_mnd = (RIOOL_PER_MND + VVE_PER_MND + ORV_PER_MND) * (1 + HEFFING_STIJGING) ** (j - 1)
    return ozb + overige_mnd * 12


def koop_kosten_cumulatief(jaren, incl_eenmalig=True):
    """Cumulatieve KOSTEN van kopen (rente + heffingen - fiscaal voordeel).
    Aflossing telt NIET mee (dat is vermogensopbouw)."""
    cum = []
    totaal = AANKOOPKOSTEN_NU + OVERDRACHTSBEL_NU if incl_eenmalig else 0.0
    for j in range(1, jaren + 1):
        netto_kosten = koop_jaarrente(j) + heffingen_jaar(j) - koop_jaarfiscaal(j)
        totaal += netto_kosten
        cum.append(totaal)
    return np.array(cum)


def overstapkosten_tweede_huis(woningwaarde_nu):
    """Eenmalige kosten van het kopen van een volgend huis na verkoop."""
    if not OVERSTAP_MEEREKENEN:
        return 0.0
    prijs_2e = TWEEDE_HUIS_PRIJS if TWEEDE_HUIS_PRIJS is not None else woningwaarde_nu
    return prijs_2e * OVERDRACHTSBEL_PCT + NIEUWE_FIN_KOSTEN


def opgebouwd_vermogen(jaren):
    """Wat je netto terugkrijgt als je in jaar j verkoopt EN een volgend huis koopt.

    OVERBIEDEN: als de woning onder marktwaarde gekocht wordt, is de startmarktwaarde
    KOOPSOM + OVERBIEDEN. Dit extra verschil groeit mee met WAARDESTIJGING en levert
    extra vermogen op bij verkoop.

        vermogen = aflossing + waardestijging - makelaarscourtage - overstapkosten
        waardestijging = (KOOPSOM + OVERBIEDEN) * (1+WAARDESTIJGING)^j - KOOPSOM
    """
    cum = []
    afgelost = 0.0
    marktwaarde_start = KOOPSOM + OVERBIEDEN
    for j in range(1, jaren + 1):
        afgelost += AFLOSSING_PER_JAAR.get(j, 0)
        woningwaarde = marktwaarde_start * (1 + WAARDESTIJGING) ** j
        waardestijging = woningwaarde - KOOPSOM
        verkoopkosten = woningwaarde * MAKELAAR_PCT
        overstap = overstapkosten_tweede_huis(woningwaarde)
        cum.append(afgelost + waardestijging - verkoopkosten - overstap)
    return np.array(cum)


## 3. Bereken en plot

We tonen drie lijnen:
- **Huur cumulatief** — al het huurgeld dat je kwijt bent.
- **Koop cumulatieve kosten** — rente + heffingen − fiscaal voordeel + eenmalige kosten (geen aflossing).
- **Koop netto kosten** — koopkosten *minus* opgebouwd vermogen (aflossing + waardestijging − makelaarscourtage − kosten tweede huis); dit is wat kopen je "echt" kost als je op dat moment zou verkopen en doorverhuizen.

In [ ]:
jaren = np.arange(1, HORIZON_JAREN + 1)

huur_cum   = huur_kosten_cumulatief(HORIZON_JAREN)
koop_cum   = koop_kosten_cumulatief(HORIZON_JAREN)
vermogen   = opgebouwd_vermogen(HORIZON_JAREN)
koop_netto = koop_cum - vermogen

fig, ax = plt.subplots(figsize=(6.5, 6.5))

ax.plot(jaren, huur_cum,   marker='o', lw=2.2, label='Huur (cumulatief verloren)')
ax.plot(jaren, koop_netto, marker='^', lw=2.2, label='Koop: netto kosten (na vermogensopbouw)')

ax.axhline(0, color='grey', lw=0.8, ls='--')

diff = huur_cum - koop_netto
for i in range(len(diff) - 1):
    if diff[i] < 0 and diff[i+1] >= 0:
        ax.axvline(jaren[i+1], color='green', ls=':', alpha=0.6)
        ax.text(jaren[i+1], ax.get_ylim()[1]*0.05,
                f'  break-even ~jaar {jaren[i+1]}', color='green', fontsize=9)
        break

ax.set_xlabel('Jaren', fontsize=12)
ax.set_ylabel('Cumulatief bedrag (euro)', fontsize=12)
ax.set_title('Huren vs. Kopen — cumulatieve kosten over de tijd\nRotterdam', fontsize=13)
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'EUR {x:,.0f}'))

plt.tight_layout()
plt.savefig('huur_vs_koop.png', dpi=130, bbox_inches='tight')
plt.show()


## 4. Tabel met getallen per jaar

In [67]:
print(f"{'Jaar':>4} | {'Huur cum.':>12} | {'Koop kosten':>12} | {'Koop netto':>12} | {'Verschil':>12}")
print('-' * 66)
for i, j in enumerate(jaren):
    verschil = huur_cum[i] - koop_netto[i]   # positief = kopen voordeliger
    print(f"{j:>4} | {huur_cum[i]:>12,.0f} | {koop_cum[i]:>12,.0f} | "
          f"{koop_netto[i]:>12,.0f} | {verschil:>+12,.0f}")
print('-' * 66)
print("Verschil positief = kopen is op dat moment voordeliger dan huren.")


Jaar |    Huur cum. |  Koop kosten |   Koop netto |     Verschil
------------------------------------------------------------------
   1 |       22,284 |       15,688 |       36,215 |      -13,931
   2 |       45,459 |       30,703 |       42,922 |       +2,538
   3 |       69,562 |       45,619 |       49,173 |      +20,388
   4 |       94,628 |       60,434 |       54,950 |      +39,678
   5 |      120,697 |       75,140 |       60,229 |      +60,468
------------------------------------------------------------------
Verschil positief = kopen is op dat moment voordeliger dan huren.


## 5. Scenario's spelen

Wijzig hierboven bijvoorbeeld:
- `WAARDESTIJGING = 0.0` — wat als de woning niet in waarde stijgt?
- `WOZ_STIJGING` — hoe hard de WOZ-waarde (en dus de OZB) jaarlijks oploopt.
- `HEFFING_STIJGING` — jaarlijkse stijging van riool, VvE en ORV.
- `VVE_PER_MND` — de grootste onbekende; vul het echte bedrag in zodra je het servicekostenoverzicht hebt.
- `HUUR_STIJGING` — bij een hogere huurverhoging wordt kopen sneller voordelig.
- `OVERSTAP_MEEREKENEN = False` — laat de kosten van een tweede huis (overdrachtsbelasting + financieringskosten) buiten beschouwing.
- `TWEEDE_HUIS_PRIJS` — vul een concrete koopsom in voor het volgende huis (None = even duur als deze woning op het verkoopmoment).

Run daarna de cellen opnieuw om de plot en tabel bij te werken.